# Demo — Busca em arquivos: o agente que procura como um dev

**Objetivo:** responder perguntas sobre uma base de código sem índice, sem embedding e sem banco vetorial. É a linha **Busca em arquivos** do slide 19 ("RAGless: recuperar sem banco vetorial, e quando isso cabe"): o agente lista pastas, abre arquivos e procura termos, como um dev numa base de código.

1. **As ferramentas na mão** → o que o agente enxerga quando lista, procura e lê, e o que acontece quando ele pede um arquivo fora do repositório;
2. **Onde está o timeout de produção?** → o agente procura o termo, abre os arquivos e separa os 8 s de produção dos 15 s do código e dos 30 s de homologação;
3. **O que fazer no erro 504?** → o termo exato leva à seção certa do runbook;
4. **Quanto custou cada pergunta** → chamadas ao modelo e tokens: o custo migra para a consulta.

> **Cenário 100% fictício.** A Contoso Industrial, o repositório `servico-pagamentos` e todos os arquivos, incidentes e números da pasta `repositorio/` existem apenas para fins didáticos.

## 0. Preparação

Este notebook usa só a **Azure OpenAI v1 API** (cliente Python `OpenAI`), com a Responses API e chamada de funções. Não há índice, embedding nem AI Search: o que o agente consulta é a pasta `repositorio/`, ao lado deste notebook, com sete arquivos de um serviço fictício de pagamento a fornecedores.

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT`, no formato `https://<recurso>.openai.azure.com`
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat que aceite chamada de funções na Responses API, por exemplo `gpt-5-mini`)

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução. A célula de configuração é a mesma dos outros notebooks: ela imprime o acervo compartilhado do Cap. 5, que aqui não é usado, e localiza a pasta `Cap05`, de onde sai o caminho do repositório.

> Não grave chaves diretamente no notebook.

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Acervo:", PASTA_ACERVO)

### O repositório e as três ferramentas

O agente não recebe o repositório: recebe **ferramentas** para explorá-lo, as mesmas que um dev usa no terminal.

- `listar_pasta(caminho=".")`: o que há numa pasta (as pastas terminam com `/`);
- `procurar_termo(termo, caminho=".")`: um "grep" que não diferencia maiúsculas e devolve até 30 linhas no formato `arquivo:linha: texto`;
- `ler_arquivo(caminho, linha_inicial=1, linha_final=80)`: as linhas pedidas, numeradas, até 80 por chamada.

Quem escolhe os argumentos é o **modelo**. Por isso todo caminho passa por `caminho_seguro`. Caminho absoluto é recusado antes de qualquer acesso ao disco: no Windows, só resolver um caminho de rede (`//servidor/...`) já tenta uma conexão com aquele servidor, e essa conexão pode expor a credencial de rede de quem roda o notebook. Os demais caminhos são resolvidos, o que desfaz `..` e atalhos, e recusados se ficarem fora de `repositorio/`. Nenhuma ferramenta escreve, apaga ou executa nada, e as mensagens de erro citam só o caminho que o modelo pediu, nunca o caminho real na máquina.

In [ ]:
REPOSITORIO = (RAIZ / "ragless" / "03-busca-em-arquivos" / "repositorio").resolve()


def caminho_seguro(caminho: str) -> Path:
    """Recusa caminho absoluto e qualquer caminho que, resolvido, fique fora do repositório."""
    # Caminho absoluto (C:/..., /etc/..., //servidor/...) é recusado antes de qualquer acesso ao disco ou à rede.
    if Path(caminho).anchor:
        raise PermissionError(f"acesso negado: '{caminho}' fica fora do repositório")
    alvo = (REPOSITORIO / caminho).resolve()  # desfaz ".." e atalhos antes de comparar
    if not alvo.is_relative_to(REPOSITORIO):
        raise PermissionError(f"acesso negado: '{caminho}' fica fora do repositório")
    if not alvo.exists():
        raise FileNotFoundError(f"'{caminho}' não existe no repositório")
    return alvo


def listar_pasta(caminho: str = ".") -> str:
    pasta = caminho_seguro(caminho)
    if pasta.is_file():
        return f"'{caminho}' é um arquivo: use ler_arquivo."
    itens = sorted(pasta.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    return "\n".join(p.name + ("/" if p.is_dir() else "") for p in itens) or "(pasta vazia)"


def procurar_termo(termo: str, caminho: str = ".") -> str:
    base = caminho_seguro(caminho)
    # Cada arquivo também é conferido: um atalho dentro do repositório poderia apontar para fora.
    arquivos = [base] if base.is_file() else sorted(
        p for p in base.rglob("*") if p.is_file() and p.resolve().is_relative_to(REPOSITORIO)
    )
    achados = []
    for arquivo in arquivos:
        nome = arquivo.relative_to(REPOSITORIO).as_posix()
        for numero, linha in enumerate(arquivo.read_text(encoding="utf-8", errors="replace").splitlines(), start=1):
            if termo.lower() in linha.lower():
                achados.append(f"{nome}:{numero}: {linha.strip()}")
    if not achados:
        return f"Nenhuma ocorrência de '{termo}'."
    resto = f"\n(... e mais {len(achados) - 30} linhas: refine o termo ou a pasta)" if len(achados) > 30 else ""
    return "\n".join(achados[:30]) + resto


def ler_arquivo(caminho: str, linha_inicial: int = 1, linha_final: int = 80) -> str:
    arquivo = caminho_seguro(caminho)
    if arquivo.is_dir():
        return f"'{caminho}' é uma pasta: use listar_pasta."
    linhas = arquivo.read_text(encoding="utf-8").splitlines()
    inicio = max(linha_inicial, 1)
    fim = min(linha_final, inicio + 79, len(linhas))
    if inicio > fim:
        return f"{caminho} tem {len(linhas)} linhas."
    numeradas = [f"{n:>3}: {linhas[n - 1]}" for n in range(inicio, fim + 1)]
    return "\n".join([f"{caminho} (linhas {inicio} a {fim} de {len(linhas)})", *numeradas])


print("Repositório:", REPOSITORIO)

### O agente: as ferramentas declaradas e o laço

O modelo nunca vê o código Python das ferramentas. Vê só a **declaração** de cada uma (nome, descrição e parâmetros em JSON Schema), enviada em toda chamada e cobrada como token de entrada. As declarações usam `strict: False`: o modelo segue o esquema no melhor esforço, e quem decide o que é aceito é o código.

O laço é o de qualquer agente:

1. o modelo recebe a pergunta e as declarações, e devolve **pedidos de ferramenta** (`function_call`) ou a resposta final;
2. o código executa cada pedido, com os argumentos que o modelo escolheu, e devolve o resultado (`function_call_output`); um erro volta como texto, para o modelo poder corrigir o pedido;
3. repete até o modelo responder sem pedir ferramenta, ou até `max_passos`.

Com `store=False`, o serviço não guarda nada entre as chamadas: a cada passo reenviamos a lista `entrada` inteira, que só cresce. Cada pedido de ferramenta volta para a lista só com `type`, `call_id`, `name` e `arguments`, **sem o `id` do item**: um pedido com `id`, mas sem o raciocínio ligado a ele, pode ser recusado pela API.

Para modelos de raciocínio, a recomendação da Microsoft é outra: acrescentar `include=["reasoning.encrypted_content"]` à chamada `responses.create` e reenviar todos os itens da resposta (`entrada += resposta.output`), inclusive o raciocínio, que no modo sem estado viaja cifrado. Assim o modelo continua de onde parou, em vez de recomeçar, e chega à resposta com menos tokens. Aqui ficamos com a forma mínima, mais simples de ler; o preço é o modelo refazer o raciocínio a cada passo. A Execução 4 mostra o consumo total, sem separar essa parcela.

Cada pedido executado imprime uma linha de rastro, e cada chamada ao modelo registra os tokens em `CONSUMO`, usado na Execução 4.

In [ ]:
import json

# O que o modelo sabe de cada ferramenta: nome, descrição e parâmetros (JSON Schema).
FERRAMENTAS = [
    {
        "type": "function",
        "name": "listar_pasta",
        "description": "Lista as pastas e os arquivos de uma pasta do repositório. As pastas terminam com '/'.",
        "parameters": {
            "type": "object",
            "properties": {
                "caminho": {"type": "string", "description": "Pasta relativa à raiz do repositório; '.' é a raiz."},
            },
            "required": [],
        },
        "strict": False,
    },
    {
        "type": "function",
        "name": "procurar_termo",
        "description": (
            "Procura um termo exato, sem diferenciar maiúsculas, nos arquivos de uma pasta do repositório "
            "e devolve até 30 linhas no formato arquivo:linha: texto."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "termo": {"type": "string", "description": "Texto literal: nome de chave, de função, código de erro."},
                "caminho": {"type": "string", "description": "Pasta ou arquivo onde procurar; '.' é o repositório todo."},
            },
            "required": ["termo"],
        },
        "strict": False,
    },
    {
        "type": "function",
        "name": "ler_arquivo",
        "description": "Lê um arquivo do repositório e devolve as linhas pedidas, numeradas, até 80 por chamada.",
        "parameters": {
            "type": "object",
            "properties": {
                "caminho": {"type": "string", "description": "Arquivo relativo à raiz, por exemplo config/producao.yaml."},
                "linha_inicial": {"type": "integer", "description": "Primeira linha, a partir de 1."},
                "linha_final": {"type": "integer", "description": "Última linha."},
            },
            "required": ["caminho"],
        },
        "strict": False,
    },
]

FUNCOES = {"listar_pasta": listar_pasta, "procurar_termo": procurar_termo, "ler_arquivo": ler_arquivo}


def executar_ferramenta(nome: str, argumentos: str) -> str:
    """Executa o pedido do modelo. Qualquer erro volta como texto, para o modelo poder corrigir o pedido."""
    try:
        return FUNCOES[nome](**json.loads(argumentos))
    except Exception as erro:
        return f"Erro: {type(erro).__name__}: {erro}"

In [ ]:
INSTRUCOES_AGENTE = """
Você responde perguntas sobre o repositório servico-pagamentos, da Contoso Industrial.
Explore o repositório com as ferramentas, como um desenvolvedor faria: liste pastas,
procure termos exatos (nomes de chave, de função, códigos de erro) e leia os trechos relevantes.
Não responda de memória: cada afirmação precisa vir de um arquivo que você leu.
Cite a fonte de cada afirmação no formato [arquivo:linha].
Se o repositório não trouxer a informação, responda: 'Não encontrei evidência no repositório.'
Responda em português, de forma objetiva.
""".strip()

CONSUMO = {}  # pergunta -> tokens de cada chamada ao modelo (Execução 4)


def executar_agente(pergunta: str, max_passos: int = 10) -> str:
    """O laço do agente: o modelo pede ferramentas e o código executa, até vir a resposta final."""
    entrada = [{"role": "user", "content": pergunta}]
    CONSUMO[pergunta] = []
    for passo in range(1, max_passos + 1):
        resposta = client.responses.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            instructions=INSTRUCOES_AGENTE,
            input=entrada,
            tools=FERRAMENTAS,
            store=False,
        )
        pedidos = [item for item in resposta.output if item.type == "function_call"]
        detalhes = resposta.usage.input_tokens_details  # se o serviço não mandar o detalhamento, conta como 0
        CONSUMO[pergunta].append({
            "passo": passo,
            "ferramentas pedidas": len(pedidos),
            "tokens de entrada": resposta.usage.input_tokens,
            "em cache": (detalhes.cached_tokens if detalhes else 0) or 0,
            "tokens de saída": resposta.usage.output_tokens,
        })
        if not pedidos:
            return resposta.output_text

        # Forma mínima: os pedidos voltam só com os campos obrigatórios, sem o id do item e sem o raciocínio.
        entrada += [
            {"type": "function_call", "call_id": p.call_id, "name": p.name, "arguments": p.arguments}
            for p in pedidos
        ]
        for p in pedidos:
            resultado = executar_ferramenta(p.name, p.arguments)
            linhas = resultado.splitlines()
            print(f"passo {passo} | {p.name} {p.arguments} → {len(linhas)} linha(s) | {linhas[0][:70]}")
            entrada.append({"type": "function_call_output", "call_id": p.call_id, "output": resultado})
    return f"O agente parou no limite de {max_passos} passos, sem chegar a uma resposta."

---

# Execução 1 — As ferramentas na mão

Antes de ligar o modelo, chamamos cada ferramenta como o agente chamaria. O texto impresso é **exatamente** o que o modelo recebe como resultado: não há trecho, score nem vetor, só nome de arquivo, número de linha e texto.

Duas chamadas a mais mostram os limites: uma procura por `tempo limite`, um sinônimo que o repositório não usa, e um pedido de `../../../.env`, fora do repositório, que é onde fica o arquivo de chaves do Cap. 5.

In [ ]:
print(">>> listar_pasta('.')")
print(listar_pasta("."))

print("\n>>> procurar_termo('timeout')")
print(procurar_termo("timeout"))

print("\n>>> procurar_termo('tempo limite')")
print(procurar_termo("tempo limite"))

print("\n>>> ler_arquivo('config/producao.yaml')")
print(ler_arquivo("config/producao.yaml"))

# O mesmo pedido que o modelo, ou um texto malicioso dentro de um arquivo, poderia fazer:
print("\n>>> ler_arquivo('../../../.env')")
print(executar_ferramenta("ler_arquivo", json.dumps({"caminho": "../../../.env"})))

### O que observar

- `procurar_termo('timeout')` devolve 20 linhas de seis arquivos, com quatro números por perto: 8, 10, 15 e 30. Quem separa um do outro é a leitura do arquivo certo, não a semelhança do texto.
- `tempo limite` não acha nada: a busca é literal. Quem traz o vocabulário do código (`timeout`, `timeout_segundos`) é o modelo.
- O pedido de `../../../.env` volta como **erro**, e o conteúdo do arquivo de chaves nunca sai do disco.

### Leitura arquitetural

- O agente executa **o seu código**, com argumentos que **o modelo escolheu**. Validar cada chamada, dar só leitura e dar o mínimo de acesso é a recomendação da Microsoft para chamada de funções.
- O conteúdo dos arquivos também é entrada não confiável: um comentário malicioso no código pode pedir ao modelo que leia o `.env`. Quem impede o vazamento não é o prompt, é a ferramenta.

---

# Execução 2 — Qual é o timeout de produção, e onde ele é definido?

A pergunta tem uma armadilha de propósito: o repositório tem **vários** timeouts. Produção, homologação, o valor padrão do código e o limite do runbook falam do mesmo assunto, com números diferentes.

### O que esperar e por que pode variar

- O agente tende a procurar `timeout` (ou `timeout_segundos`) e depois abrir `config/producao.yaml` e `src/pagamentos/gateway.py`.
- O caminho muda de uma execução para outra (quais termos, quantos passos, em que ordem), porque é o modelo que decide. A conclusão não deveria mudar: **8 s**, na chave `timeout_segundos` de `config/producao.yaml`; os 15 s de `TIMEOUT_PADRAO` só valem se a chave faltar; os 30 s são de homologação.
- Se a resposta misturar os números, leia o rastro: qual arquivo ele abriu, e qual deixou de abrir.

In [ ]:
pergunta_timeout = "Qual é o timeout das chamadas ao gateway de pagamento em produção, e onde ele é definido?"
display(Markdown(f"**Pergunta:** {pergunta_timeout}"))

resposta_timeout = executar_agente(pergunta_timeout)
display(Markdown(resposta_timeout))

### O que observar

- O rastro mostra a recuperação acontecendo **na hora da pergunta**: cada linha é um passo em que o modelo escolheu uma ferramenta e um argumento.
- A resposta cita `[arquivo:linha]`, e cada número pode ser conferido abrindo o arquivo: é a rastreabilidade que o auditor pede, sem índice nenhum.
- Uma busca por semelhança tenderia a trazer, com a mesma confiança, os trechos que **falam** de timeout: a ADR, o runbook, a homologação. Aqui quem decide é a chave exata `timeout_segundos`, no arquivo de produção.

---

# Execução 3 — O que o runbook manda fazer no erro 504?

O runbook tem uma seção por código de erro, todas escritas com o mesmo modelo de texto: 401, 422, "502 ou 503" e 504. Para um embedding, essas seções são quase iguais, como os SKUs 48213-B e 48213-C do slide 14. Para a busca literal, `504` é outro termo.

### O que esperar e por que pode variar

- O agente tende a procurar `504`, às vezes depois de listar a pasta `docs/`, e abrir a seção do erro 504.
- A resposta deve trazer os passos dessa seção: página de status do gateway, incidente de severidade 2 acima de 5% de erros por mais de 10 minutos, **não aumentar o timeout acima de 10 s sem aprovação** e reprocessar só as ordens com `FALHA_GATEWAY`.
- Se ela misturar a seção de 502 e 503 (pausar o lote na manutenção programada), o rastro mostra onde: qual intervalo de linhas ele leu.

In [ ]:
pergunta_504 = "O que o runbook manda fazer quando o gateway devolve erro 504?"
display(Markdown(f"**Pergunta:** {pergunta_504}"))

resposta_504 = executar_agente(pergunta_504)
display(Markdown(resposta_504))

### O que observar

- Procurado no repositório todo, `504` aparece na seção certa do runbook, na ADR e em `retentativas.py`, onde está entre os códigos que o serviço repete sozinho. Juntar runbook e código é trabalho do agente, passo a passo.
- Se o agente lê só um intervalo de linhas, e não o arquivo inteiro, a conta agradece: cada linha devolvida vira token de entrada em todos os passos seguintes.

### Fala sugerida

> “Em documentação técnica, 504 não é parecido com 503: é outro procedimento. Quem decide é o termo exato.”

---

# Execução 4 — Quanto custou cada pergunta

Não houve indexação: nenhum embedding, nenhum índice, nada pago antes da primeira pergunta. Em compensação, cada pergunta custou **várias** chamadas ao modelo, e cada chamada reenviou as instruções, as declarações das ferramentas, a pergunta e tudo o que as ferramentas já tinham devolvido.

A coluna **em cache** é o prompt caching do Azure OpenAI: quando os primeiros 1.024 tokens ou mais de um prompt são idênticos aos de uma chamada recente, essa parte sai com desconto. O laço só acrescenta itens no fim da lista, que é o formato que o cache favorece. Em perguntas curtas, a coluna pode ficar em zero.

Essa é a regra dos modelos do GPT-4o até a família GPT-5.5, como o `gpt-5-mini`. Da família GPT-5.6 em diante, o cache funciona por pontos de corte (*breakpoints*), como explica o `../01-contexto-longo`: os números desta coluna mudam, e gravar no cache pode ser cobrado.

In [ ]:
display(pd.DataFrame([
    {
        "pergunta": pergunta,
        "chamadas ao modelo": len(passos),
        "ferramentas executadas": sum(p["ferramentas pedidas"] for p in passos),
        "tokens de entrada": sum(p["tokens de entrada"] for p in passos),
        "dos quais em cache": sum(p["em cache"] for p in passos),
        "tokens de saída": sum(p["tokens de saída"] for p in passos),
    }
    for pergunta, passos in CONSUMO.items()
]))

display(Markdown(f"**Passo a passo da Execução 2:** {pergunta_timeout}"))
display(pd.DataFrame(CONSUMO[pergunta_timeout]))

### O que observar

- No passo a passo, os **tokens de entrada** crescem a cada chamada: cada passo paga de novo tudo o que os anteriores trouxeram.
- Em modelos de raciocínio, como os da família gpt-5, os tokens de saída incluem os de raciocínio: são cobrados como saída mesmo quando o texto final é curto. Como o laço não reenvia o raciocínio, o modelo raciocina de novo a cada passo, e cada passo paga esses tokens outra vez.
- Os números variam com o modelo, com o caminho que o agente escolheu e com o cache. O que importa é a ordem de grandeza: compare com uma única chamada de RAG clássico, com 3 a 5 trechos no contexto (slide 9).

### Leitura arquitetural

- **O custo migra para a consulta** (slide 19). O RAG clássico paga a indexação uma vez, e cada pergunta custa uma busca e uma chamada ao modelo. Aqui não há indexação, e cada pergunta paga várias chamadas.
- Quem segura a conta é o desenho das ferramentas: limite de linhas por busca e por leitura, e resultado curto, porque cada resultado é reenviado em todos os passos seguintes.
- Em modelos de raciocínio, a Microsoft recomenda reenviar também os itens de raciocínio, cifrados no modo sem estado: o modelo continua de onde parou e chega à resposta com menos tokens. Os números acima são os da forma mínima, sem esse reenvio.
- Limites de escala: a busca percorre todos os arquivos a cada chamada, cada pergunta vira uma pequena investigação, e o agente enxerga tudo o que o processo enxerga, não só o que o usuário que perguntou pode ver (slide 26). Como diz o rodapé do slide 19: "Com acervo grande, volátil ou com permissão por usuário, o índice volta a valer."

---

# Quadro final — o que cada execução provou?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | As ferramentas devolvem arquivo, linha e texto; a busca é literal e o caminho fora do repositório é recusado | Ferramentas só de leitura, com o caminho validado no código, nunca no prompt |
| 2 | O agente separa 8 s, 15 s e 30 s abrindo os arquivos certos, e cita `arquivo:linha` | Busca em arquivos quando o termo exato decide e a fonte precisa ser conferível |
| 3 | `504` leva à seção certa entre seções quase iguais | Em código e documentação técnica, o identificador vale mais que a semelhança |
| 4 | Várias chamadas ao modelo por pergunta, com entrada que cresce a cada passo | O custo migra para a consulta: limitar passos e o tamanho de cada resultado |

## Frase de fechamento

> **Código e documentação técnica, onde o termo exato decide.**

---

# Referências oficiais

- Azure OpenAI — Responses API, com chamada de funções, uso sem estado (store=false) e raciocínio cifrado:  
  https://learn.microsoft.com/azure/foundry/openai/how-to/responses

- Azure OpenAI — modelos de raciocínio (tokens de raciocínio cobrados como saída; reenvio do raciocínio junto com as chamadas de função):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/reasoning

- Azure OpenAI — chamada de funções e uso responsável (validar chamadas, menor privilégio):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/function-calling

- Azure OpenAI — prompt caching (1.024 tokens, cached_tokens):  
  https://learn.microsoft.com/azure/foundry/openai/how-to/prompt-caching